# Caduceus (SSM) Geometric Stability Scaling Experiment

Tests the architectural hypothesis: **Do SSMs show the same instability-with-scale as Transformers?**

## Key Difference

- **ESM-2 (Transformer)**: Attention mechanism, protein sequences
- **Caduceus (SSM/Mamba)**: State-space model, DNA sequences

If SSMs don't show geometric instability with scale, this proves it's an architectural property of Transformers, not just a general scaling phenomenon.

## Important Note on Scale Range

Available Caduceus checkpoints span a limited parameter range (~0.5M to ~4M),
compared to ESM-2's nearly four orders of magnitude (8M to 15B). The key comparison
is the **slope**: if ESM-2 already degrades between 8M and 35M while Caduceus stays
flat across its range, the architectural difference is clear.

## Setup Instructions

1. Upload `evaluation_harness.py` and `perturbation_protocol.py`
2. Change Runtime to GPU (Runtime > Change runtime type > GPU)
3. Run cells in order

---

In [ ]:
# Install Dependencies
#
# Strategy: Build mamba-ssm from source to match Colab's CUDA/PyTorch.
# Prebuilt cu11 wheels are ABI-incompatible with Colab's CUDA 12 PyTorch.
# --no-build-isolation ensures it uses the already-installed torch/CUDA.
#
# This takes ~10-15 min on first run but only happens once per session.
# Go get coffee. Once built, everything loads natively with full pretrained
# weights -- no shim, no missing keys, no NaN.

print("Installing core dependencies...")
!pip install -q torch transformers shesha-geometry matplotlib seaborn pandas einops

# Clean any stale cu11 binaries that cause ABI mismatch
print("\nCleaning stale mamba installs...")
!pip uninstall -y mamba-ssm selective-scan-cuda causal-conv1d 2>/dev/null; true

print("\nBuilding causal-conv1d from source...")
!pip install causal-conv1d --no-build-isolation

print("\nBuilding mamba-ssm from source (this takes ~10-15 min, one-time cost)...")
!pip install mamba-ssm --no-build-isolation

# Verify
import torch
print(f"\ntorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

import mamba_ssm
print(f"mamba-ssm {mamba_ssm.__version__} -- native CUDA kernels OK!")
from mamba_ssm.modules.mamba_simple import Mamba
print("Mamba class imported successfully. No shim needed.")

print("\nDone! Native mamba-ssm is ready.")


In [ ]:
# Configuration

PHASE = 'full'  # 'quick' or 'full'

import os
import sys
sys.path.insert(0, '.')
OUTPUT_BASE = './results/caduceus_scaling_experiment/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR = OUTPUT_BASE + 'cache'

# Use caduceus-PS (Parameter Sharing) models -- these are RC-equivariant.
# PS models have rcps=True natively: the RCPSWrapper flips inputs for the
# reverse branch and ties weights, giving true Reverse Complement equivariance.
# PH models (caduceus-ph_*) are bidirectional but NOT RC-equivariant.
CONFIG = {
    # Quick validation: test if SSMs behave differently
    'quick': {
        'n_sequences': 500,
        'seq_length': 1000,  # DNA nucleotides (Caduceus context: up to 131k)
        'models': [
            'kuleshov-group/caduceus-ps_seqlen-1k_d_model-118_n_layer-4_lr-8e-3',   # 471k
            'kuleshov-group/caduceus-ps_seqlen-131k_d_model-256_n_layer-16',        # 7.73M
        ],
        'batch_size': 8,
        'n_bootstrap': 0,
        'snp_rates': [0.01, 0.05, 0.10],
    },
    # Full scaling sweep: all available Caduceus sizes
    'full': {
        'n_sequences': 10000,
        'seq_length': 1000,
        'models': [
            'kuleshov-group/caduceus-ps_seqlen-1k_d_model-118_n_layer-4_lr-8e-3',   # 471k
            'kuleshov-group/caduceus-ps_seqlen-1k_d_model-256_n_layer-4_lr-8e-3',   # 1.93M
            'kuleshov-group/caduceus-ps_seqlen-131k_d_model-256_n_layer-16',        # 7.73M
        ],
        'batch_size': 4,
        'n_bootstrap': 5,
        'snp_rates': [0.01, 0.02, 0.05, 0.10],
    },
}

config = CONFIG[PHASE]
print(f"Phase: {PHASE.upper()}")
print(f"Architecture: Caduceus (Mamba/SSM)")
print(f"Sequences: {config['n_sequences']}")
print(f"Sequence length: {config['seq_length']} nucleotides")
print(f"Models: {len(config['models'])}")

In [ ]:
# Generate DNA Sequences
#
# IMPORTANT: Caduceus is a DNA model (uses ACGT nucleotides)

import numpy as np

DNA_BASES = ['A', 'C', 'G', 'T']

def generate_dna_sequences(n_sequences, seq_length, seed=320):
    """Generate random DNA sequences for Caduceus."""
    rng = np.random.default_rng(seed)
    sequences = [
        ''.join(rng.choice(DNA_BASES, size=seq_length))
        for _ in range(n_sequences)
    ]
    print(f"Generated {len(sequences)} DNA sequences (length={seq_length})")
    print(f"Sample: {sequences[0][:50]}...")
    return sequences

sequences = generate_dna_sequences(config['n_sequences'], config['seq_length'])

In [ ]:
# DNA Perturbation Suite

from dataclasses import dataclass, field

COMPLEMENT = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}

@dataclass
class PerturbedSet:
    name: str
    category: str
    sequences: list
    params: dict = field(default_factory=dict)
    description: str = ''


def mutate_dna(sequence, mutation_rate, rng):
    """SNP mutations at the given rate."""
    seq = list(sequence)
    n_mutations = max(1, int(len(seq) * mutation_rate))
    positions = rng.choice(len(seq), size=n_mutations, replace=False)
    for pos in positions:
        original = seq[pos]
        alternatives = [b for b in DNA_BASES if b != original]
        seq[pos] = rng.choice(alternatives)
    return ''.join(seq)


def reverse_complement(sequence):
    """Reverse complement (biologically equivalent for DNA)."""
    return ''.join(COMPLEMENT.get(b, b) for b in reversed(sequence))


class DNAPerturbationSuite:
    """DNA perturbation suite for Caduceus."""

    def __init__(self, seed=320, snp_rates=None, include_rc=True):
        self.rng = np.random.default_rng(seed)
        self.snp_rates = snp_rates or [0.01, 0.02, 0.05, 0.10]
        self.include_rc = include_rc

    def run_all(self, sequences):
        results = {}

        # SNP mutations
        for rate in self.snp_rates:
            name = f"snp_{int(rate * 100)}pct"
            perturbed = [mutate_dna(s, rate, self.rng) for s in sequences]
            results[name] = PerturbedSet(
                name=name,
                category='snp',
                sequences=perturbed,
                params={'mutation_rate': rate},
                description=f'SNP mutations at {rate*100:.0f}% rate',
            )

        # Reverse complement
        if self.include_rc:
            name = 'reverse_complement'
            perturbed = [reverse_complement(s) for s in sequences]
            results[name] = PerturbedSet(
                name=name,
                category='rc',
                sequences=perturbed,
                params={},
                description='Reverse complement transformation',
            )

        return results

    def get_perturbation_names(self):
        names = [f"snp_{int(r*100)}pct" for r in self.snp_rates]
        if self.include_rc:
            names.append('reverse_complement')
        return names

    def summary(self):
        lines = [
            'DNA Perturbation Protocol',
            '=' * 40,
            f'SNP rates: {self.snp_rates}',
            f'Reverse complement: {self.include_rc}',
            f'Total perturbation sets: {len(self.get_perturbation_names())}',
        ]
        return '\n'.join(lines)


suite = DNAPerturbationSuite(seed=320, snp_rates=config['snp_rates'])
print(suite.summary())

test_perturbed = suite.run_all(sequences[:5])
for name, pset in test_perturbed.items():
    dists = [
        sum(a != b for a, b in zip(o, p)) / len(o)
        for o, p in zip(sequences[:5], pset.sequences)
    ]
    print(f"  {name}: mean_hamming={np.mean(dists):.4f}")
print("Perturbation suite ready")

In [ ]:
# Caduceus Model Wrapper
#
# Native mamba-ssm only -- no pure-Python shim.
# builds mamba-ssm from source, so native CUDA kernels are available.
# This gives full pretrained weights, no missing keys, no NaN.
#
# CRITICAL: Caduceus "ph" (Parameter sHaring) models require TWO things:
#   1. Shared weights between mamba_fwd and mamba_rev  (weight tying)
#   2. config.rcps = True  (activates the geometric FLIP on inputs/outputs)
# Without #2, both branches see the same un-flipped input and the model
# is effectively unidirectional -- destroying RC equivariance entirely.

import os
import torch
import gc
from transformers import AutoModel, AutoTokenizer, AutoConfig
import transformers.modeling_utils as _mu


def _patch_tied_weights():
    """Patch transformers for Caduceus compatibility.

    Caduceus's tie_weights() doesn't define all_tied_weights_keys or accept
    the missing_keys/recompute_mapping kwargs that newer transformers expects.
    This is needed even with native mamba-ssm.
    """
    if getattr(_mu, '_caduceus_patched', False):
        return

    orig_mark = _mu.PreTrainedModel.mark_tied_weights_as_initialized

    def safe_mark(self):
        if not hasattr(self, 'all_tied_weights_keys'):
            self.all_tied_weights_keys = {}
        return orig_mark(self)

    _mu.PreTrainedModel.mark_tied_weights_as_initialized = safe_mark

    orig_finalize = _mu.PreTrainedModel._finalize_load_state_dict

    @staticmethod
    def safe_finalize(model, load_config, load_info):
        if not hasattr(model, 'all_tied_weights_keys'):
            model.all_tied_weights_keys = {}
        orig_tie = model.tie_weights
        def _patched_tie(missing_keys=None, recompute_mapping=False, **kwargs):
            return orig_tie()
        model.tie_weights = _patched_tie
        return orig_finalize(model, load_config, load_info)

    _mu.PreTrainedModel._finalize_load_state_dict = safe_finalize
    _mu._caduceus_patched = True
    print("  Patched transformers for Caduceus compatibility")

_patch_tied_weights()


def cleanup_gpu():
    """Free GPU memory between models."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        mem = torch.cuda.memory_allocated() / 1e9
        print(f"  GPU memory after cleanup: {mem:.2f} GB")


def make_caduceus_embedding_fn(model_name, batch_size=4):
    """Load Caduceus model with native mamba-ssm and return embedding function."""
    print(f"Loading {model_name}...")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"  Device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    # Load config -- respect the checkpoint's native architecture.
    # IMPORTANT: PH vs PS are fundamentally different architectures:
    #   PH ("Pre-trained Human"): bidirectional=True, rcps=False
    #       -> Two separate Mamba branches, NO geometric flip, NOT RC-equivariant
    #   PS ("Parameter Sharing"):  bidirectional=True, rcps=True
    #       -> RCPSWrapper flips input for reverse branch, weight-tied, RC-equivariant
    # DO NOT force rcps=True on PH checkpoints -- it wraps layers in RCPSWrapper
    # which adds .submodule. to all paths, causing a complete state_dict mismatch.
    cfg = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
    is_rcps = getattr(cfg, 'rcps', False)
    print(f"  Config: rcps={is_rcps}, bidirectional={getattr(cfg, 'bidirectional', 'N/A')}")
    if not is_rcps:
        print("  NOTE: This is a PH (non-RCPS) model. It will NOT show RC equivariance.")
        print("        Use caduceus-ps_* models for RC equivariant experiments.")

    # Disable fused Triton norms if they're not available
    try:
        from mamba_ssm.ops.triton.layer_norm import layer_norm_fn, rms_norm_fn
        if layer_norm_fn is None or rms_norm_fn is None:
            raise ImportError("fused norms are None")
    except (ImportError, AttributeError):
        if hasattr(cfg, 'fused_add_norm'):
            cfg.fused_add_norm = False
            print("  Disabled fused_add_norm (Triton norms not available)")
        if hasattr(cfg, 'fused_dropout_add_ln'):
            cfg.fused_dropout_add_ln = False

    model = AutoModel.from_pretrained(
        model_name, config=cfg, trust_remote_code=True,
    ).to(device).eval()

    # Re-tie mamba_rev weights to mamba_fwd after checkpoint loading.
    # from_pretrained can break weight ties by assigning new tensors.
    if hasattr(model, 'tie_weights'):
        model.tie_weights()
        print("  Re-tied mamba_rev <-> mamba_fwd weights after loading")

    # Verify weight ties (path differs by architecture)
    # PS (rcps=True): layers wrap in RCPSWrapper -> .mixer.submodule.mamba_fwd
    # PH (rcps=False): direct access            -> .mixer.mamba_fwd
    try:
        layer0_mixer = model.backbone.layers[0].mixer
        inner = layer0_mixer.submodule if hasattr(layer0_mixer, 'submodule') else layer0_mixer
        fwd_ptr = inner.mamba_fwd.in_proj.weight.data_ptr()
        rev_ptr = inner.mamba_rev.in_proj.weight.data_ptr()
        tied = fwd_ptr == rev_ptr
        print(f"  Weight tie verification: {'OK - same tensor' if tied else 'SEPARATE tensors (expected for PH, broken for PS)'}")
        if is_rcps and not tied:
            print("  WARNING: PS model but weights not tied! Forcing re-tie...")
            for layer in model.backbone.layers:
                mix = layer.mixer.submodule if hasattr(layer.mixer, 'submodule') else layer.mixer
                mix.mamba_rev.in_proj.weight = mix.mamba_fwd.in_proj.weight
                if hasattr(mix.mamba_fwd.in_proj, 'bias') and mix.mamba_fwd.in_proj.bias is not None:
                    mix.mamba_rev.in_proj.bias = mix.mamba_fwd.in_proj.bias
                mix.mamba_rev.out_proj.weight = mix.mamba_fwd.out_proj.weight
                if hasattr(mix.mamba_fwd.out_proj, 'bias') and mix.mamba_fwd.out_proj.bias is not None:
                    mix.mamba_rev.out_proj.bias = mix.mamba_fwd.out_proj.bias
            fwd2 = inner.mamba_fwd.in_proj.weight.data_ptr()
            rev2 = inner.mamba_rev.in_proj.weight.data_ptr()
            print(f"  After manual re-tie: {'OK' if fwd2 == rev2 else 'STILL BROKEN'}")
    except Exception as e:
        print(f"  Weight tie check skipped: {e}")

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    if torch.cuda.is_available():
        mem = torch.cuda.memory_allocated() / 1e9
        print(f"  Params: {n_params:.1f}M | GPU mem: {mem:.2f} GB")
    else:
        print(f"  Params: {n_params:.1f}M")

    @torch.no_grad()
    def embed(sequences):
        embeddings = []
        n_batches = (len(sequences) + batch_size - 1) // batch_size

        for i in range(0, len(sequences), batch_size):
            batch = sequences[i:i + batch_size]
            batch_idx = i // batch_size + 1

            if batch_idx % 20 == 0 or batch_idx == n_batches:
                print(f"    Batch {batch_idx}/{n_batches}", end='\r')

            tokens = tokenizer(
                batch, return_tensors='pt', padding=True,
                truncation=True, max_length=131072,
            )
            tokens = {k: v.to(device) for k, v in tokens.items()}

            outputs = model(**tokens, output_hidden_states=True)
            hidden = outputs.hidden_states[-1]

            # RCPS models have doubled hidden dim: [fwd_half; rev_half].
            # For seq s the model outputs [a; b], for rc(s) it outputs [b; a].
            # Averaging the halves gives RC-INVARIANT embeddings: (a+b)/2 == (b+a)/2.
            if is_rcps:
                d = hidden.shape[-1]
                half1 = hidden[..., :d//2]
                half2 = hidden[..., d//2:]
                hidden = (half1 + half2.flip(dims=[-1])) / 2

            if 'attention_mask' in tokens:
                mask = tokens['attention_mask'].unsqueeze(-1)
                pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            else:
                pooled = hidden.mean(dim=1)
            embeddings.append(pooled.cpu().numpy())

        print()
        return np.concatenate(embeddings, axis=0)

    return embed, model, tokenizer, n_params

print("Caduceus wrapper ready (native mamba-ssm)")

In [ ]:
# Set Up Evaluation Harness

from evaluation_harness import StabilityHarness

harness = StabilityHarness(
    window_size=0,
    metric='cosine',
    n_splits=30,
    seed=320,
    max_samples=2500,
    n_bootstrap=config['n_bootstrap'],
)

print(f"Harness configured (bootstrap={config['n_bootstrap']})")

In [ ]:
# Run Experiment

import os
import time
import pandas as pd
from evaluation_harness import ModelReport

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print('=' * 70)
print(f'CADUCEUS (SSM) SCALING EXPERIMENT -- Phase: {PHASE.upper()}')
print('=' * 70)

reports = []
model_param_counts = []  # Track actual param counts
all_detailed_rows = []   # Per-perturbation results

for model_idx, model_name in enumerate(config['models']):
    print(f"\n{'='*70}")
    print(f"[{model_idx+1}/{len(config['models'])}] {model_name}")
    print('=' * 70)

    start_time = time.time()

    # Load model -- now returns actual param count
    embed_fn, model_obj, tokenizer_obj, n_params_m = make_caduceus_embedding_fn(
        model_name, config['batch_size']
    )
    model_param_counts.append(n_params_m)

    # Generate perturbations
    print('\nGenerating perturbations...')
    perturbed_sets = suite.run_all(sequences)

    # Compute clean embeddings (with caching)
    safe_name = model_name.replace('/', '_').replace('-', '_')
    cache_clean = f'{CACHE_DIR}/{safe_name}_clean.npy'

    if os.path.exists(cache_clean):
        print(f'Loading cached clean embeddings: {cache_clean}')
        embeddings_clean = np.load(cache_clean)
    else:
        print(f'Computing clean embeddings ({len(sequences)} sequences)...')
        embeddings_clean = embed_fn(sequences)
        np.save(cache_clean, embeddings_clean)
        print(f'  Cached to {cache_clean}')
    print(f'  Shape: {embeddings_clean.shape}')

    # Compute perturbed embeddings
    perturbed_embeddings = {}
    for pert_name, pset in perturbed_sets.items():
        cache_pert = f'{CACHE_DIR}/{safe_name}_{pert_name}.npy'
        if os.path.exists(cache_pert):
            print(f'  Loading cached: {pert_name}')
            perturbed_embeddings[pert_name] = np.load(cache_pert)
        else:
            print(f'  Embedding: {pert_name}...')
            perturbed_embeddings[pert_name] = embed_fn(pset.sequences)
            np.save(cache_pert, perturbed_embeddings[pert_name])

    # Free GPU memory
    del model_obj, tokenizer_obj
    cleanup_gpu()

    # Run Shesha evaluation
    print('\nRunning Shesha evaluation...')
    report = harness.evaluate_all_perturbations(
        model_name=model_name,
        embeddings_clean=embeddings_clean,
        perturbed_dict=perturbed_embeddings,
    )
    reports.append(report)

    # Collect per-perturbation rows
    short_name = model_name.split('/')[-1]
    for pert_name, metrics in report.perturbation_breakdown().items():
        all_detailed_rows.append({
            'model': short_name,
            'size_M': round(n_params_m, 1),
            'perturbation': pert_name,
            'rdm_similarity': metrics.get('rdm_similarity_score', 0),
            'rdm_drift': metrics.get('rdm_drift', 0),
            'pert_stability': metrics.get('perturbation_stability_score', 0),
            'pert_magnitude': metrics.get('perturbation_magnitude', 0),
            'composite': metrics.get('composite_stability', 0),
        })

    elapsed = time.time() - start_time
    summary = report.summary()
    print(f'\nCompleted in {elapsed/60:.1f} min')
    print(f'  Composite Stability: {summary["mean_composite_stability"]:.4f}')
    print(f'  RDM Similarity:      {summary["mean_rdm_similarity_score"]:.4f}')
    print(f'  Pert. Stability:     {summary["mean_perturbation_stability_score"]:.4f}')

# Save detailed per-perturbation CSV
df_detailed = pd.DataFrame(all_detailed_rows)
detailed_path = f'{RESULTS_DIR}/caduceus_scaling_{PHASE}_detailed.csv'
df_detailed.to_csv(detailed_path, index=False)
print(f'\nPer-perturbation results:')
print(df_detailed.to_string(index=False))
print(f'\nSaved to {detailed_path}')

print('\n' + '=' * 70)
print('EXPERIMENT COMPLETE')
print('=' * 70)

In [ ]:
# Visualize with Actual Parameter Counts

import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 12})

summaries = [r.summary() for r in reports]
model_names = [r.model_name for r in reports]

# Use actual param counts (not hardcoded approximations)
sizes = model_param_counts

metrics = {
    'mean_composite_stability': ('Composite Stability', 'tab:blue'),
    'mean_rdm_similarity_score': ('RDM Similarity', 'tab:green'),
    'mean_perturbation_stability_score': ('Perturbation Stability', 'tab:purple'),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (metric_key, (label, color)) in enumerate(metrics.items()):
    ax = axes[idx]
    values = [s[metric_key] for s in summaries]
    std_key = metric_key.replace('mean_', 'std_')
    errors = [s.get(std_key, 0) for s in summaries]

    ax.errorbar(sizes, values, yerr=errors, fmt='o-', color=color,
                linewidth=2, markersize=10, capsize=5)
    ax.set_xlabel('Parameters (M)')
    ax.set_ylabel(label)
    ax.set_title(label, fontweight='bold')
    ax.grid(True, alpha=0.3)
    if max(sizes) / max(min(sizes), 0.01) > 10:
        ax.set_xscale('log')

fig.suptitle(
    'Caduceus (SSM) Geometric Stability vs. Model Size',
    fontsize=14, fontweight='bold', y=1.02,
)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/caduceus_scaling_{PHASE}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'Saved to results/caduceus_scaling_{PHASE}.png')

In [ ]:
# Cross-Experiment Comparison (SSM vs Transformer)
#
# This is the key plot: overlay Caduceus scaling with ESM-2 scaling
# to show the architectural difference.

import matplotlib.pyplot as plt
import pandas as pd
import glob

# Try to load ESM-2 synthetic results
esm2_files = glob.glob(f'{RESULTS_DIR}/esm2_scaling_*_detailed.csv')
if not esm2_files:
    esm2_files = glob.glob('../**/esm2_scaling_*_detailed.csv', recursive=True)

if esm2_files:
    print(f'Found ESM-2 results: {esm2_files[0]}')
    df_esm2 = pd.read_csv(esm2_files[0])

    # Aggregate by model
    esm2_agg = df_esm2.groupby(['model', 'size_M'])['composite'].mean().reset_index()
    cad_agg = df_detailed.groupby(['model', 'size_M'])['composite'].mean().reset_index()

    fig, ax = plt.subplots(figsize=(10, 6))

    # ESM-2 (Transformer)
    ax.plot(esm2_agg['size_M'], esm2_agg['composite'], 'o-',
            color='tab:red', linewidth=2, markersize=10,
            label='ESM-2 (Transformer)', zorder=3)

    # Caduceus (SSM)
    ax.plot(cad_agg['size_M'], cad_agg['composite'], 's-',
            color='tab:blue', linewidth=2, markersize=10,
            label='Caduceus (SSM)', zorder=3)

    ax.set_xscale('log')
    ax.set_xlabel('Parameters (M)', fontsize=12)
    ax.set_ylabel('Composite Stability', fontsize=12)
    ax.set_title('The Geometric Tax: Transformer vs. SSM Scaling', fontweight='bold', fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

    # Annotate the slope difference
    ax.annotate('Stability degrades with scale',
                xy=(esm2_agg['size_M'].iloc[-1], esm2_agg['composite'].iloc[-1]),
                xytext=(20, 20), textcoords='offset points',
                arrowprops=dict(arrowstyle='->', color='tab:red'),
                fontsize=10, color='tab:red', fontstyle='italic')

    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/transformer_vs_ssm_scaling_{PHASE}.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved comparison plot')

    # Print the slope comparison
    print('\n--- Slope Analysis ---')
    if len(esm2_agg) >= 2:
        esm2_slope = (esm2_agg['composite'].iloc[-1] - esm2_agg['composite'].iloc[0])
        print(f'ESM-2 composite change: {esm2_slope:+.4f} (first to last)')
    if len(cad_agg) >= 2:
        cad_slope = (cad_agg['composite'].iloc[-1] - cad_agg['composite'].iloc[0])
        print(f'Caduceus composite change: {cad_slope:+.4f} (first to last)')
else:
    print('No ESM-2 results found.')
    print('Run the ESM-2 scaling experiment first, or copy its results CSV to results/')
    print('Expected filename pattern: results/esm2_scaling_*_detailed.csv')